In [5]:
#!/usr/bin/env python3
"""
Resistor-ladder reconstruction from EITKit logs.

Features:
- Parse ORIGIN_DATA / FRAME_DATA from serial logs.
- Keep only complete modal-length frames.
- Select settled frames (manual start/end or automatic warm-up detection).
- Fit masked rank-1 model M[tx, rx] ~= k_tx * r_rx.
- Normalize by tx_scale before final edge-profile fit.
- Compute bootstrap confidence intervals per edge.
- Export CSV, summary, and PNG plots (if matplotlib is available).
"""

from __future__ import annotations

import argparse
import csv
import math
import os
import random
import re
import statistics
from collections import Counter
from dataclasses import dataclass
from typing import List, Optional, Sequence, Tuple


try:
    import matplotlib.pyplot as plt  # type: ignore

    HAVE_MPL = True
except Exception:
    HAVE_MPL = False


FLOAT_RE = re.compile(r"[-+]?(?:\d+\.\d*|\.\d+|\d+)")


@dataclass
class ParsedLog:
    origin: Optional[List[float]]
    frames: List[List[float]]
    dropped_frames: int
    frame_len: int


def parse_numeric_payload(text: str) -> List[float]:
    return [float(m.group(0)) for m in FLOAT_RE.finditer(text)]


def parse_log_file(path: str) -> ParsedLog:
    origin_candidates: List[List[float]] = []
    frame_candidates: List[List[float]] = []

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if line.startswith("ORIGIN_DATA,"):
                vals = parse_numeric_payload(line.split(",", 1)[1])
                if vals:
                    origin_candidates.append(vals)
            elif line.startswith("FRAME_DATA,"):
                vals = parse_numeric_payload(line.split(",", 1)[1])
                if vals:
                    frame_candidates.append(vals)

    if not frame_candidates:
        raise ValueError("No FRAME_DATA lines found in log.")

    lengths = Counter(len(v) for v in frame_candidates)
    frame_len, _ = lengths.most_common(1)[0]

    frames = [v for v in frame_candidates if len(v) == frame_len]
    dropped = len(frame_candidates) - len(frames)
    if not frames:
        raise ValueError("No complete modal-length frames available.")

    origin: Optional[List[float]] = None
    for candidate in origin_candidates:
        if len(candidate) == frame_len:
            origin = candidate
            break

    return ParsedLog(origin=origin, frames=frames, dropped_frames=dropped, frame_len=frame_len)


def infer_num_electrodes(frame_len: int) -> int:
    n = int(round(math.sqrt(frame_len)))
    if n * n != frame_len:
        raise ValueError(
            f"Frame length {frame_len} is not a perfect square; cannot infer electrode count."
        )
    return n


def reshape_flat(flat: List[float], n: int) -> List[List[float]]:
    return [flat[i * n : (i + 1) * n] for i in range(n)]


def flatten_matrix(m: List[List[float]]) -> List[float]:
    out: List[float] = []
    for row in m:
        out.extend(row)
    return out


def adjacent_ad_mask(num_elec: int) -> List[List[bool]]:
    # AD drive: src=tx, sink=(tx+1)%N
    # AD measure: vp=rx, vn=(rx+1)%N
    # invalid if measurement touches drive electrodes.
    mask = [[True for _ in range(num_elec)] for _ in range(num_elec)]
    for tx in range(num_elec):
        src = tx
        sink = (tx + 1) % num_elec
        for rx in range(num_elec):
            vp = rx
            vn = (rx + 1) % num_elec
            invalid = (vp == src) or (vp == sink) or (vn == src) or (vn == sink)
            if invalid:
                mask[tx][rx] = False
    return mask


def aggregate_frames(frames: List[List[float]], stat: str) -> List[float]:
    frame_len = len(frames[0])
    out = [0.0] * frame_len
    for i in range(frame_len):
        col = [fr[i] for fr in frames]
        out[i] = statistics.median(col) if stat == "median" else statistics.mean(col)
    return out


def masked_rank1_fit(
    m: List[List[float]], mask: List[List[bool]], max_iter: int = 200, eps: float = 1e-12
) -> Tuple[List[float], List[float], List[List[float]], float]:
    n_tx = len(m)
    n_rx = len(m[0])

    r = [1.0] * n_rx
    for rx in range(n_rx):
        vals = [m[tx][rx] for tx in range(n_tx) if mask[tx][rx]]
        if vals:
            r[rx] = max(statistics.median(vals), eps)

    k = [1.0] * n_tx

    for _ in range(max_iter):
        r_prev = r[:]

        for tx in range(n_tx):
            num = 0.0
            den = 0.0
            for rx in range(n_rx):
                if mask[tx][rx]:
                    num += m[tx][rx] * r[rx]
                    den += r[rx] * r[rx]
            if den > eps:
                k[tx] = max(num / den, eps)

        for rx in range(n_rx):
            num = 0.0
            den = 0.0
            for tx in range(n_tx):
                if mask[tx][rx]:
                    num += m[tx][rx] * k[tx]
                    den += k[tx] * k[tx]
            if den > eps:
                r[rx] = max(num / den, eps)

        norm_prev = math.sqrt(sum(v * v for v in r_prev))
        norm_delta = math.sqrt(sum((a - b) * (a - b) for a, b in zip(r, r_prev)))
        rel = norm_delta / max(norm_prev, eps)
        if rel < 1e-8:
            break

    fit = [[k[tx] * r[rx] for rx in range(n_rx)] for tx in range(n_tx)]

    valid_meas: List[float] = []
    valid_fit: List[float] = []
    for tx in range(n_tx):
        for rx in range(n_rx):
            if mask[tx][rx]:
                valid_meas.append(m[tx][rx])
                valid_fit.append(fit[tx][rx])

    mean_meas = statistics.mean(valid_meas) if valid_meas else 0.0
    ss_res = sum((a - b) * (a - b) for a, b in zip(valid_meas, valid_fit))
    ss_tot = sum((a - mean_meas) * (a - mean_meas) for a in valid_meas)
    r2 = 1.0 - (ss_res / ss_tot) if ss_tot > eps else float("nan")

    # Normalize r to median=1
    med = statistics.median(r)
    if med > eps:
        r = [x / med for x in r]
        k = [x * med for x in k]
        fit = [[k[tx] * r[rx] for rx in range(n_rx)] for tx in range(n_tx)]

    return r, k, fit, r2


def normalize_rows(m: List[List[float]], tx_scale: List[float], eps: float = 1e-12) -> List[List[float]]:
    med = statistics.median(tx_scale)
    out: List[List[float]] = []
    for tx, row in enumerate(m):
        s = tx_scale[tx]
        if abs(s) < eps:
            gain = 1.0
        else:
            gain = med / s
        out.append([v * gain for v in row])
    return out


def frame_active_mean(frame: List[float], active_idx: List[int]) -> float:
    vals = [frame[i] for i in active_idx]
    return statistics.mean(vals)


def auto_detect_settled_start(
    frame_means: List[float], window: int = 30, tol_rel: float = 0.02
) -> int:
    # Baseline from tail median; settle when window stays within tol.
    n = len(frame_means)
    if n < max(10, window + 5):
        return 0

    tail = frame_means[max(0, int(0.7 * n)) :]
    baseline = statistics.median(tail)
    if baseline == 0:
        return 0

    for i in range(0, n - window):
        ok = True
        for j in range(i, i + window):
            rel = abs(frame_means[j] - baseline) / abs(baseline)
            if rel > tol_rel:
                ok = False
                break
        if ok:
            return i
    return 0


def quantile(vals: List[float], q: float) -> float:
    if not vals:
        return float("nan")
    xs = sorted(vals)
    if len(xs) == 1:
        return xs[0]
    pos = q * (len(xs) - 1)
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))
    if lo == hi:
        return xs[lo]
    t = pos - lo
    return xs[lo] * (1.0 - t) + xs[hi] * t


def bootstrap_edge_ci(
    frames: List[List[float]],
    n: int,
    mask: List[List[bool]],
    frame_stat: str,
    n_boot: int,
    alpha: float,
    seed: int,
) -> Tuple[List[float], List[float]]:
    rng = random.Random(seed)
    boot: List[List[float]] = []
    f_count = len(frames)
    if f_count == 0:
        return [float("nan")] * n, [float("nan")] * n

    for _ in range(n_boot):
        sample = [frames[rng.randrange(f_count)] for _ in range(f_count)]
        avg_flat = aggregate_frames(sample, stat=frame_stat)
        avg_m = reshape_flat(avg_flat, n)
        _, k_tx, _, _ = masked_rank1_fit(avg_m, mask)
        avg_norm = normalize_rows(avg_m, k_tx)
        r_norm, _, _, _ = masked_rank1_fit(avg_norm, mask)
        boot.append(r_norm)

    lo_q = alpha / 2.0
    hi_q = 1.0 - alpha / 2.0
    ci_lo = []
    ci_hi = []
    for edge in range(n):
        vals = [b[edge] for b in boot]
        ci_lo.append(quantile(vals, lo_q))
        ci_hi.append(quantile(vals, hi_q))
    return ci_lo, ci_hi


def summarize_frames(
    frames: List[List[float]], mask_flat: List[bool], dropped_frames: int, num_elec: int
) -> dict:
    active_idx = [i for i, v in enumerate(mask_flat) if v]
    per_frame_mean: List[float] = []
    per_frame_std: List[float] = []
    for fr in frames:
        vals = [fr[i] for i in active_idx]
        m = statistics.mean(vals)
        per_frame_mean.append(m)
        var = sum((x - m) * (x - m) for x in vals) / len(vals)
        per_frame_std.append(math.sqrt(var))

    return {
        "num_frames": len(frames),
        "num_channels": len(frames[0]),
        "active_channels": sum(mask_flat),
        "zero_channels": len(mask_flat) - sum(mask_flat),
        "frame_mean_min": min(per_frame_mean),
        "frame_mean_max": max(per_frame_mean),
        "frame_mean_median": statistics.median(per_frame_mean),
        "frame_std_median": statistics.median(per_frame_std),
        "inferred_electrodes": num_elec,
        "dropped_incomplete_frames": dropped_frames,
    }


def write_vector_csv(path: str, header: Sequence[str], rows: List[Sequence[object]]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(list(header))
        for row in rows:
            w.writerow(list(row))


def write_matrix_csv(path: str, m: List[List[float]]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        for row in m:
            w.writerow([f"{v:.10f}" for v in row])


def save_plots(
    out_dir: str,
    frame_means_all: List[float],
    settled_start: int,
    settled_end: int,
    avg_m: List[List[float]],
    fit_m: List[List[float]],
    mask: List[List[bool]],
    edge_r: List[float],
    ci_lo: List[float],
    ci_hi: List[float],
) -> None:
    if not HAVE_MPL:
        return

    n = len(avg_m)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(frame_means_all, linewidth=1.2)
    ax.axvline(settled_start, color="green", linestyle="--", linewidth=1.2, label="settled start")
    if settled_end < len(frame_means_all) - 1:
        ax.axvline(settled_end, color="orange", linestyle="--", linewidth=1.2, label="settled end")
    ax.set_title("Frame Mean Over Time (active channels)")
    ax.set_xlabel("Frame index")
    ax.set_ylabel("Mean amplitude (V rms)")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "frame_mean_timeseries.png"), dpi=180)
    plt.close(fig)

    # Mask invalids as NaN for display.
    disp = []
    for tx in range(n):
        row = []
        for rx in range(n):
            row.append(avg_m[tx][rx] if mask[tx][rx] else float("nan"))
        disp.append(row)

    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(disp, aspect="auto", interpolation="nearest")
    ax.set_title("Average Measurement Matrix (tx x rx)")
    ax.set_xlabel("rx edge index")
    ax.set_ylabel("tx pair index")
    fig.colorbar(im, ax=ax, label="V rms")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "avg_measurement_matrix.png"), dpi=180)
    plt.close(fig)

    resid = []
    for tx in range(n):
        row = []
        for rx in range(n):
            row.append((avg_m[tx][rx] - fit_m[tx][rx]) if mask[tx][rx] else float("nan"))
        resid.append(row)

    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(resid, aspect="auto", interpolation="nearest", cmap="coolwarm")
    ax.set_title("Residual Matrix (avg - rank1 fit)")
    ax.set_xlabel("rx edge index")
    ax.set_ylabel("tx pair index")
    fig.colorbar(im, ax=ax, label="Residual (V rms)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "fit_residual_matrix.png"), dpi=180)
    plt.close(fig)

    idx = list(range(n))
    err_lo = [max(0.0, edge_r[i] - ci_lo[i]) for i in range(n)]
    err_hi = [max(0.0, ci_hi[i] - edge_r[i]) for i in range(n)]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(idx, edge_r, color="#2C7FB8")
    ax.errorbar(idx, edge_r, yerr=[err_lo, err_hi], fmt="none", ecolor="black", capsize=3, lw=1)
    ax.axhline(1.0, linestyle="--", linewidth=1.4, color="black", alpha=0.7)
    ax.set_title("Reconstructed Relative Edge Resistance Profile")
    ax.set_xlabel("edge index (between node i and i+1)")
    ax.set_ylabel("relative resistance (median=1)")
    ax.set_xticks(idx)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "edge_resistance_profile.png"), dpi=180)
    plt.close(fig)


def main(argv: Optional[Sequence[str]] = None) -> int:
    parser = argparse.ArgumentParser(description="Reconstruct resistor-ladder profile from EIT logs.")
    parser.add_argument("log_file", help="Path to updated_logs.txt")
    parser.add_argument("--out-dir", default="output_results", help="Output directory")
    parser.add_argument(
        "--frame-stat",
        choices=("mean", "median"),
        default="median",
        help="Aggregation statistic across settled frames before fitting",
    )

    # Settled-frame controls
    parser.add_argument(
        "--settled-start",
        type=int,
        default=120,
        help="Manual settled start frame index (default: 120)",
    )
    parser.add_argument(
        "--settled-end",
        type=int,
        default=-1,
        help="Manual settled end frame index inclusive (-1 means last frame)",
    )
    parser.add_argument(
        "--auto-settle",
        action="store_true",
        help="Auto-detect settled start using frame-mean stability",
    )
    parser.add_argument("--auto-window", type=int, default=30, help="Auto-settle stable window size")
    parser.add_argument(
        "--auto-tol-rel",
        type=float,
        default=0.02,
        help="Auto-settle relative tolerance vs tail baseline (default: 0.02 = 2%)",
    )

    # Normalization + confidence intervals
    parser.add_argument(
        "--tx-normalize",
        action="store_true",
        default=True,
        help="Normalize rows by tx_scale before final edge fit (enabled by default)",
    )
    parser.add_argument("--no-tx-normalize", action="store_true", help="Disable tx normalization")
    parser.add_argument("--ci-bootstrap", type=int, default=300, help="Bootstrap samples for edge CI")
    parser.add_argument("--ci-alpha", type=float, default=0.05, help="Two-sided CI alpha (default 0.05)")
    parser.add_argument("--seed", type=int, default=1337, help="Random seed for bootstrap")
    parser.add_argument(
        "--tx-anomaly-threshold",
        type=float,
        default=0.03,
        help="Flag tx rows with |scale/median-1| above this fraction (default 0.03 = 3%)",
    )
    args = parser.parse_args(argv)

    parsed = parse_log_file(args.log_file)
    n = infer_num_electrodes(parsed.frame_len)
    mask = adjacent_ad_mask(n)
    mask_flat = flatten_matrix(mask)
    active_idx = [i for i, v in enumerate(mask_flat) if v]

    all_frame_means = [frame_active_mean(fr, active_idx) for fr in parsed.frames]

    settled_start_warning = ""
    if args.settled_start >= len(parsed.frames):
        manual_start = 0
        settled_start_warning = (
            f"requested settled_start={args.settled_start} exceeds available frames "
            f"({len(parsed.frames)}); using 0 instead"
        )
    else:
        manual_start = max(0, args.settled_start)
    if args.auto_settle:
        auto_start = auto_detect_settled_start(
            all_frame_means, window=max(5, args.auto_window), tol_rel=max(1e-6, args.auto_tol_rel)
        )
        settled_start = max(manual_start, auto_start)
    else:
        settled_start = manual_start

    if args.settled_end < 0:
        settled_end = len(parsed.frames) - 1
    else:
        settled_end = min(args.settled_end, len(parsed.frames) - 1)
    if settled_end < settled_start:
        settled_end = settled_start

    settled_frames = parsed.frames[settled_start : settled_end + 1]
    if not settled_frames:
        raise ValueError("No settled frames selected.")

    # Pre-normalization fit
    avg_flat_pre = aggregate_frames(settled_frames, stat=args.frame_stat)
    avg_m_pre = reshape_flat(avg_flat_pre, n)
    edge_pre, tx_scale, _, r2_pre = masked_rank1_fit(avg_m_pre, mask)

    use_tx_norm = args.tx_normalize and (not args.no_tx_normalize)
    if use_tx_norm:
        avg_m = normalize_rows(avg_m_pre, tx_scale)
    else:
        avg_m = avg_m_pre

    edge_r, tx_scale_post, fit_m, r2_post = masked_rank1_fit(avg_m, mask)
    resid_m = [[avg_m[tx][rx] - fit_m[tx][rx] for rx in range(n)] for tx in range(n)]
    _ = tx_scale_post  # not used directly; kept for debugging expansion.

    # Bootstrap CI on final pipeline.
    if args.ci_bootstrap > 0:
        ci_lo, ci_hi = bootstrap_edge_ci(
            frames=settled_frames,
            n=n,
            mask=mask,
            frame_stat=args.frame_stat,
            n_boot=args.ci_bootstrap,
            alpha=args.ci_alpha,
            seed=args.seed,
        )
    else:
        ci_lo = [float("nan")] * n
        ci_hi = [float("nan")] * n

    # Anomaly detection on tx_scale from pre-normalization fit.
    tx_med = statistics.median(tx_scale)
    tx_pct_dev = [((v / tx_med) - 1.0) if tx_med != 0 else 0.0 for v in tx_scale]
    tx_flags = [abs(d) > args.tx_anomaly_threshold for d in tx_pct_dev]
    flagged_rows = [i for i, flag in enumerate(tx_flags) if flag]

    summary = summarize_frames(
        frames=settled_frames,
        mask_flat=mask_flat,
        dropped_frames=parsed.dropped_frames,
        num_elec=n,
    )
    summary["settled_start"] = settled_start
    summary["settled_end"] = settled_end
    summary["settled_frames"] = len(settled_frames)
    summary["used_tx_normalization"] = use_tx_norm
    summary["rank1_fit_r2_pre_norm"] = round(r2_pre, 6)
    summary["rank1_fit_r2_post_norm"] = round(r2_post, 6)
    summary["tx_anomaly_threshold_frac"] = args.tx_anomaly_threshold
    summary["tx_anomaly_rows"] = flagged_rows
    summary["origin_present"] = parsed.origin is not None
    summary["matplotlib_available"] = HAVE_MPL
    if settled_start_warning:
        summary["settled_start_warning"] = settled_start_warning

    os.makedirs(args.out_dir, exist_ok=True)

    # CSV outputs
    write_vector_csv(
        os.path.join(args.out_dir, "edge_profile.csv"),
        ("edge_index", "relative_resistance", "ci_low", "ci_high"),
        [
            (
                i,
                f"{edge_r[i]:.10f}",
                f"{ci_lo[i]:.10f}" if not math.isnan(ci_lo[i]) else "",
                f"{ci_hi[i]:.10f}" if not math.isnan(ci_hi[i]) else "",
            )
            for i in range(n)
        ],
    )
    write_vector_csv(
        os.path.join(args.out_dir, "tx_scale.csv"),
        ("tx_index", "scale", "pct_dev_from_median", "anomaly_flag", "src_electrode", "sink_electrode"),
        [
            (
                tx,
                f"{tx_scale[tx]:.10f}",
                f"{100.0 * tx_pct_dev[tx]:.6f}",
                int(tx_flags[tx]),
                tx,
                (tx + 1) % n,
            )
            for tx in range(n)
        ],
    )
    write_matrix_csv(os.path.join(args.out_dir, "avg_matrix.csv"), avg_m)
    write_matrix_csv(os.path.join(args.out_dir, "fit_matrix.csv"), fit_m)
    write_matrix_csv(os.path.join(args.out_dir, "residual_matrix.csv"), resid_m)

    # Human summary
    with open(os.path.join(args.out_dir, "summary.txt"), "w", encoding="utf-8") as f:
        f.write("Reconstruction Summary\n")
        f.write("======================\n")
        for k, v in summary.items():
            f.write(f"{k}: {v}\n")
        f.write("\n")
        f.write("Interpretation Notes\n")
        f.write("====================\n")
        if flagged_rows:
            f.write(
                "Detected tx_scale anomalies. Probe contact/wiring/mux path for these tx pairs:\n"
            )
            for tx in flagged_rows:
                src = tx
                sink = (tx + 1) % n
                pct = 100.0 * tx_pct_dev[tx]
                f.write(f"  tx={tx} (src={src}, sink={sink}), scale dev={pct:+.2f}%\n")
        else:
            f.write("No tx_scale anomalies above threshold.\n")

    # PNG plots
    save_plots(
        out_dir=args.out_dir,
        frame_means_all=all_frame_means,
        settled_start=settled_start,
        settled_end=settled_end,
        avg_m=avg_m,
        fit_m=fit_m,
        mask=mask,
        edge_r=edge_r,
        ci_lo=ci_lo,
        ci_hi=ci_hi,
    )

    # Console output
    print("Reconstruction complete")
    print(f"  settled_frame_range: [{settled_start}, {settled_end}]")
    print(f"  settled_frames: {len(settled_frames)} / total {len(parsed.frames)}")
    print(f"  inferred_electrodes: {n}")
    print(f"  active_channels: {summary['active_channels']}")
    print(f"  zero_channels: {summary['zero_channels']}")
    print(f"  rank1_fit_r2_pre_norm: {r2_pre:.6f}")
    print(f"  rank1_fit_r2_post_norm: {r2_post:.6f}")
    print(f"  tx_anomaly_rows: {flagged_rows}")
    print(f"  output_dir: {os.path.abspath(args.out_dir)}")
    if settled_start_warning:
        print(f"  warning: {settled_start_warning}")
    if not HAVE_MPL:
        print("  note: matplotlib not available; PNG plots were not generated in this environment.")

    return 0



In [7]:
main([
    "/content/codex-logs.txt",
    "--out-dir", "/content/output_results_manual",
    "--settled-start", "120",
    "--settled-end", "-1",
    "--tx-normalize",
    "--ci-bootstrap", "400",
    "--ci-alpha", "0.05"
])

Reconstruction complete
  settled_frame_range: [120, 526]
  settled_frames: 407 / total 527
  inferred_electrodes: 16
  active_channels: 208
  zero_channels: 48
  rank1_fit_r2_pre_norm: 0.680108
  rank1_fit_r2_post_norm: 0.112660
  tx_anomaly_rows: [11]
  output_dir: /content/output_results_manual


0

In [8]:
main([
    "/content/codex-logs.txt",
    "--out-dir", "/content/output_results_automatic",
    "--auto-settle",
    "--auto-window", "30",
    "--auto-tol-rel", "0.02",
    "--tx-normalize",
    "--ci-bootstrap", "400"
])

Reconstruction complete
  settled_frame_range: [136, 526]
  settled_frames: 391 / total 527
  inferred_electrodes: 16
  active_channels: 208
  zero_channels: 48
  rank1_fit_r2_pre_norm: 0.680005
  rank1_fit_r2_post_norm: 0.110101
  tx_anomaly_rows: [11]
  output_dir: /content/output_results_automatic


0